# 40 Evaluate (CPU runtime)

Runtime: **CPU**. Scores a predictions file with EX, EX-tied, R-VES, VES and cost metrics, and appends one row per run to `586/effsql/results/results.csv`.
For stage 2 this notebook runs the sanity checks: gold vs gold, the official-preset runtime, and our port vs the real BIRD scripts.

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
!effsql stage --split dev

## 1. Gold vs gold sanity checks

Gold SQL scored as if it were a prediction. Expected: EX = 100, relative cost = 1, quick time ratio near 1.
The first command also fills the gold cost cache (`586/effsql/cache/gold_costs/`) and reports how many gold queries are not `ok`.

In [ ]:
!effsql evaluate --pred gold --split dev --timing none --overwrite --run sanity_gold_dev

In [ ]:
!effsql evaluate --pred gold --split minidev --timing quick --overwrite --run sanity_gold_minidev_quick

## 2. Official preset on 50 mini-dev questions (runtime check)

Exact port of the official R-VES timing (100 iterations per question, sequential). The log prints an ETA first.
Note: gold vs gold sits on the R-VES bucket boundary (ratio near 1 gives reward 1 or 0.75 by noise), so R-VES here lands between about 87 and 100.

In [ ]:
!effsql evaluate --pred gold --split minidev --timing official --limit 50 --overwrite --run sanity_gold_minidev_official50

## 3. Cross-check against the real BIRD scripts

Same 50 predictions, scored by `bird-bench/mini_dev` evaluation code at the pinned commit. EX must match exactly; R-VES should be close (timing noise).
If `evaluation_ex.py` crashes with ZeroDivisionError, the first 50 questions lack one difficulty level; rerun both with a larger `--limit`.

In [ ]:
!effsql export-official --pred gold --split minidev --limit 50 --out /content/official_check
!pip install -q func_timeout psycopg2-binary pymysql
!rm -rf /content/mini_dev && git clone -q https://github.com/bird-bench/mini_dev.git /content/mini_dev && git -C /content/mini_dev checkout -q 1c49a375e76c471e218564da3f4617513d33e371

In [ ]:
%cd /content/mini_dev/evaluation
!python3 -u evaluation_ex.py --db_root_path /content/bird_local/dev/ --predicted_sql_path /content/official_check/predict.json --ground_truth_path /content/official_check/gold.sql --diff_json_path /content/official_check/diff.jsonl --num_cpus 1 --meta_time_out 30.0 --sql_dialect SQLite --output_log_path /content/official_check/official_log.txt
!python3 -u evaluation_ves.py --db_root_path /content/bird_local/dev/ --predicted_sql_path /content/official_check/predict.json --ground_truth_path /content/official_check/gold.sql --diff_json_path /content/official_check/diff.jsonl --num_cpus 1 --meta_time_out 30.0 --sql_dialect SQLite --output_log_path /content/official_check/official_log.txt
%cd /content

## 4. Experiment ledger

In [ ]:
!cat {DRIVE_ROOT}/effsql/results/results.csv